# Tier 3 (v2) — Does the node-ZO advantage survive to a per-token backbone?

The MNIST tiers (0–2) showed node perturbation beats weight perturbation because its effective dimension is $\bar d=(1-\beta)r+\beta m\ll P=r(m+n)$. But those adapters were effectively **single-token** linear layers. A real diffusion backbone adapts **per-token / per-position** layers (attention projections, DiT MLPs), where node perturbation must probe every one of the $T$ token positions — so its effective dimension becomes $T\cdot\bar d$.

**This notebook tests, honestly, whether the advantage transfers.** It fixes the two failure modes of the first Tier-3 attempt:

1. **Real headroom, guarded by a hard assert.** The task shift is strong enough that the backprop-LoRA control measurably beats the frozen base, and the notebook `assert`s `floor < base − ε` so a headroom-free run *fails loudly* instead of reporting `L = L = L`.
2. **The budget is priced along $\beta(t)$, not at a reference.** At the standard LoRA init $B=0$ we have $\beta=1\Rightarrow\bar d=m$ (no advantage); the advantage, if any, is a property of the *trained* regime. We track $\beta(t)$ and integrate the probe budget over it.

**Registered prediction (state before running).** Node ZO wins iff $T\cdot\bar d < P=r(m+n)$. With token multiplicity $T$ and realistic $\beta$, this can *fail* — node ZO can be strictly worse than weight ZO. We predict, and then measure, exactly where the crossover sits.

> Verified end-to-end offline (numpy+torch). Runs on a Colab GPU; `FAST=True` default is a ~few-minute smoke run.


## Setup and config

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
torch.set_default_dtype(torch.float64)
dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)
FAST = True
# problem size
T, d_in, H, r = 16, 8, 32, 4        # tokens, dim/token, width, LoRA rank
Tsched = 50
# budgets
pre_steps  = 3000 if not FAST else 3000
lora_steps = 1500 if not FAST else 800
fit_reps   = 800  if not FAST else 300
Ms_fit     = [8, 32, 128]
mu         = 3e-3                    # calibrate below; float64 keeps this clean
target_a   = 0.9
print("device:", dev, "| FAST:", FAST, "| T,H,r =", (T,H,r), "| P=r(m+n) =", r*(H+H))

## Data and the shifted task

Low-rank token data (each example is $T$ tokens of dim `d_in`). The shifted task adds a strong rank-$\rho$ per-token perturbation — strong enough to create adapter headroom.

In [ ]:
klat, Ndata, rho = 4, 1024, 4
Btok = torch.randn(T, d_in, klat, device=dev)/klat**0.5
def gen(N, shift=None, s=0.0):
    c = torch.randn(N, T, klat, device=dev); x = torch.einsum('tik,ntk->nti', Btok, c)
    if shift is not None: x = x + s*torch.einsum('tik,ntk->nti', shift, torch.randn(N,T,shift.shape[-1],device=dev))
    return x
X0 = gen(Ndata); X0 = (X0-X0.mean())/X0.std()
Sh = torch.randn(T, d_in, rho, device=dev)/rho**0.5
Xs = gen(Ndata, shift=Sh, s=2.0); Xs = (Xs-Xs.mean())/Xs.std()
betas = torch.linspace(1e-4, 0.02, Tsched, device=dev); abar = torch.cumprod(1-betas, 0)
print("data ready | base and shifted (rho =", rho, ")")

## A token denoiser with a per-token adapted site

Embed each token, mix tokens with one (frozen) self-attention block, then a **per-token linear site `W2`** (the LoRA-adapted layer, $H\to H$), a nonlinearity, and a per-token head. The forward is split so we can inject perturbed node activations (`z_ovr`, `u_ovr`) at the adapted site — node perturbation, forward passes only. Because `W2` is applied to each of the $T$ tokens, the node sites are $(T,r)$ and $(T,H)$: **token multiplicity $T$**.

In [ ]:
def P_(*s): return (torch.randn(*s, device=dev)/s[-1]**0.5).requires_grad_()
W1=P_(H,d_in); b1=torch.zeros(H,device=dev,requires_grad=True); Wt=P_(H,1)
Wq=P_(H,H); Wk=P_(H,H); Wv=P_(H,H); Wo=P_(H,H)
W2=P_(H,H); Cc=P_(d_in,H)                       # W2 = adapted per-token site
base=[W1,b1,Wt,Wq,Wk,Wv,Wo,W2,Cc]
def to_hin(xt, tn):                              # frozen part up to the adapted site
    h = torch.tanh(xt@W1.T + b1 + tn[:,None,None]*Wt.T)
    q,k,v = h@Wq.T, h@Wk.T, h@Wv.T
    att = torch.softmax((q@k.transpose(1,2))/np.sqrt(H), dim=2)
    return h + (att@v)@Wo.T
def from_F(hin, A, B, z_ovr=None, u_ovr=None):
    z = hin@A.T if z_ovr is None else z_ovr
    u = z  @B.T if u_ovr is None else u_ovr
    return torch.tanh(hin@W2.T + u)@Cc.T
def mb(data, bs):
    i=torch.randint(0,len(data),(bs,),device=dev); t=torch.randint(0,Tsched,(bs,),device=dev); e=torch.randn(bs,T,d_in,device=dev)
    return abar[t].sqrt()[:,None,None]*data[i]+(1-abar[t]).sqrt()[:,None,None]*e, t.double()/Tsched, e
def dloss(data, A, B, bs=128, fixed=None):
    xt,tn,e = fixed if fixed else mb(data,bs)
    return 0.5*((from_F(to_hin(xt,tn), A, B) - e)**2).mean()
print("denoiser ready")

## Pretrain the base, then freeze (with a convergence guard)

In [ ]:
A0=torch.zeros(1,H,device=dev); B0=torch.zeros(H,1,device=dev)
opt=torch.optim.Adam(base, lr=1e-3)
for s in range(pre_steps):
    opt.zero_grad(); dloss(X0,A0,B0).backward(); opt.step()
for p in base: p.requires_grad_(False)
Lbase = dloss(X0,A0,B0,bs=512).item()
print(f"pretrain base loss = {Lbase:.4f}  (predict-zero=0.5)")
assert Lbase < 0.48, "base did not converge; raise pre_steps / lower lr before trusting the rest"
print("ASSERT PASSED: base converged")

## Real headroom — the hard guard

Train a backprop-LoRA control on the shifted task and require it to beat the frozen base by a margin. If not, the shift created no learnable signal and everything downstream is meaningless — so we `assert` and stop.

In [ ]:
Ls0 = dloss(Xs,A0,B0,bs=512).item()
def train_lora(steps, lr=2e-3, track=False):
    A=(torch.randn(r,H,device=dev)/H**0.5).requires_grad_(); B=torch.zeros(H,r,device=dev,requires_grad=True)
    o=torch.optim.Adam([A,B],lr=lr); bt=[]
    for s in range(steps):
        o.zero_grad(); dloss(Xs,A,B).backward()
        if track and s % max(1,steps//8)==0:
            EA=(A.grad**2).sum().item(); EB=(B.grad**2).sum().item(); bt.append((s, EB/(EA+EB)))
        o.step()
    return A.detach(), B.detach(), dloss(Xs,A,B,bs=512).item(), bt
Ac, Bc, floor, beta_traj = train_lora(lora_steps, track=True)
print(f"frozen base on shift = {Ls0:.4f}")
print(f"backprop-LoRA floor  = {floor:.4f}   (captured {Ls0-floor:+.4f})")
EPS = 0.015
assert floor < Ls0 - EPS, f"NO HEADROOM: control only captured {Ls0-floor:.4f} < {EPS}; strengthen the shift or the site"
print(f"ASSERT PASSED: real headroom (captured > {EPS})")

## $\beta(t)$ and the init caveat

At the standard LoRA init $B=0$, all gradient energy is in $B$, so $\beta=1$ and $\bar d=m$ — **no node advantage at the start**. We track $\beta(t)$ through training and tabulate $\beta$ (hence $\bar d$ and the token-scaled $D_\text{node}=T\bar d$) across three adapter states, with the win/lose verdict vs. weight's $P$.

In [ ]:
Pw = r*(H+H)
print("beta(t) during backprop-LoRA:", [(s, round(b,3)) for s,b in beta_traj])
def beta_of(A,B):
    fixed=mb(Xs,1); A=A.clone().requires_grad_(); B=B.clone().requires_grad_()
    dloss(Xs,A,B,fixed=fixed).backward()
    EA=(A.grad**2).sum().item(); EB=(B.grad**2).sum().item(); b=EB/(EA+EB)
    return b, (1-b)*r + b*H
print(f"\n{'adapter state':38s} {'beta':>6} {'dbar':>6} {'D_node=T*dbar':>13} {'vs P':>6}  verdict")
states=[("random both-nonzero", torch.randn(r,H,device=dev)/H**0.5, torch.randn(H,r,device=dev)/r**0.5),
        ("standard LoRA init (B=0)", torch.randn(r,H,device=dev)/H**0.5, torch.zeros(H,r,device=dev)),
        ("trained (from B=0)", Ac, Bc)]
for name,A,B in states:
    b,dbar=beta_of(A,B); Dn=T*dbar
    print(f"{name:38s} {b:>6.3f} {dbar:>6.1f} {Dn:>13.1f} {Pw:>6}  {'NODE WINS' if Dn<Pw else 'node LOSES'}")
print(f"\ncrossover: node wins iff T*dbar < P={Pw}, i.e. dbar < P/T = {Pw/T:.1f}  (i.e. beta < {(Pw/T-r)/(H-r):.3f})")

## Validate the token-multiplicity: $D_\text{node}\gtrsim T\bar d$, $D_\text{weight}\approx P$

Measure both effective dimensions at the trained state via the alignment-law fit ($1/\cos^2\theta = 1 + (D+1)/M$). $D_\text{weight}$ lands on $P$ exactly. For the node site, $T\bar d$ is only the *uncorrelated-token proxy*: the exact per-token node dimension is a token-Gram-corrected quantity (derived and verified in `Tier3_extension_token_correlation.ipynb`), which on this attention-coupled site runs ~1.15–1.25× **above** $T\bar d$. So read $T\bar d$ as a floor here; the node cost is if anything worse, which only strengthens the verdict below.

In [ ]:
# finite-mu calibration against the true directional derivative
def calibrate():
    fixed=mb(Xs,1); xt,tn,e=fixed
    A=Ac.clone().requires_grad_(); B=Bc.clone().requires_grad_()
    L=dloss(Xs,A,B,fixed=fixed); L.backward(); g=torch.cat([A.grad.flatten(),B.grad.flatten()])
    with torch.no_grad():
        u=torch.randn_like(g); uA=u[:r*H].reshape(r,H); uB=u[r*H:].reshape(H,r)
        jvp=(u@g).item()
        for m_ in [1e-2,3e-3,1e-3]:
            dd=((dloss(Xs,A+m_*uA,B+m_*uB,fixed=fixed)-dloss(Xs,A-m_*uA,B-m_*uB,fixed=fixed))/(2*m_)).item()
            print(f"  mu={m_:.0e}: (dd - jvp)/|jvp| = {(dd-jvp)/abs(jvp):+.3f}")
print("finite-mu check (dd should match the autograd directional derivative):"); calibrate()

def measure_D(A,B):
    fixed=mb(Xs,1); xt,tn,e=fixed
    A=A.clone().requires_grad_(); B=B.clone().requires_grad_()
    dloss(Xs,A,B,fixed=fixed).backward()
    g=torch.cat([A.grad.flatten(),B.grad.flatten()]); gn2=(g@g).item()
    with torch.no_grad():
        hin=to_hin(xt,tn); z0=hin@A.t(); u0=z0@B.t()
        def Lf(A_=A,B_=B,z=None,u=None): return 0.5*((from_F(hin,A_,B_,z_ovr=z,u_ovr=u)-e)**2).mean()
        def wp():
            xa=torch.randn_like(A); xb=torch.randn_like(B)
            d=(Lf(A+mu*xa,B+mu*xb)-Lf(A-mu*xa,B-mu*xb))/(2*mu)
            return torch.cat([(d*xa).flatten(),(d*xb).flatten()])
        def npb():
            xz=torch.randn_like(z0); gz=((Lf(z=z0+mu*xz)-Lf(z=z0-mu*xz))/(2*mu))*xz
            xu=torch.randn_like(u0); gu=((Lf(u=u0+mu*xu)-Lf(u=u0-mu*xu))/(2*mu))*xu
            eA=torch.einsum('tr,tn->rn',gz[0],hin[0]); eB=torch.einsum('tm,tr->mr',gu[0],z0[0])
            return torch.cat([eA.flatten(),eB.flatten()])
        def D(pr):
            inv=[]
            for M in Ms_fit:
                num=den=0.0
                for _ in range(fit_reps):
                    acc=torch.zeros_like(g)
                    for _ in range(M): acc=acc+pr()
                    acc/=M; num+=(acc@g).item(); den+=(acc@acc).item()
                inv.append(gn2*den/fit_reps/(num/fit_reps)**2)
            return np.polyfit([1/M for M in Ms_fit], inv, 1)[0]-1
        return D(npb), D(wp)
b_tr,dbar_tr = beta_of(Ac,Bc)
Dn_meas, Dw_meas = measure_D(Ac,Bc)
print(f"\nnode   D_eff measured = {Dn_meas:8.1f}  vs  T*dbar = {T*dbar_tr:7.1f}")
print(f"weight D_eff measured = {Dw_meas:8.1f}  vs  P      = {Pw:7d}")
print(f"-> node/weight cost ratio = {Dn_meas/Dw_meas:.2f}  ({'node cheaper' if Dn_meas<Dw_meas else 'node MORE expensive'})")

## The crossover, as a function of rank

Node wins iff $T\bar d < P=r(m+n)$. Sweeping rank $r$ at fixed $T$: $P$ grows as $r(m+n)$ while $\bar d$ (at fixed $\beta$) grows slowly, so large enough $r$ flips node back to winning. We plot the predicted $D_\text{node}=T\bar d$ and $P$ using the measured trained-state $\beta$, and mark the crossover.

In [ ]:
rr = np.array([2,4,8,16,32,64])
# use the trained-state beta as representative; dbar(r)=(1-b)r+b*H
b_use = b_tr
dbar_r = (1-b_use)*rr + b_use*H
Dnode_r = T*dbar_r
P_r = rr*(H+H)
plt.figure(figsize=(6.5,4))
plt.plot(rr, Dnode_r, "s-", color="C0", label=f"node  T*dbar  (beta={b_use:.2f})")
plt.plot(rr, P_r, "o-", color="C3", label="weight  P=r(m+n)")
cross = rr[np.argmax(Dnode_r < P_r)] if np.any(Dnode_r<P_r) else None
if cross: plt.axvline(cross, ls="--", color="gray", label=f"crossover ~ r={cross}")
plt.yscale("log"); plt.xscale("log"); plt.xlabel("LoRA rank r"); plt.ylabel("effective dimension")
plt.title(f"Node vs weight at a per-token site (T={T})"); plt.legend(fontsize=8); plt.grid(alpha=.3,which="both")
plt.tight_layout(); plt.show()
print("node wins (T*dbar < P) for ranks:", [int(x) for x in rr[Dnode_r<P_r]])

## Budget priced along $\beta(t)$

The online node budget is $M_n(t)=\tfrac{a}{1-a}(T\bar d(t)+1)$. Pricing at a *reference* $\beta$ under-counts the true integrated budget **only when $\beta$ actually drops** across training. At this per-token site $\beta$ stays high the whole way (see $\beta(t)$ above), so the trajectory and reference pricings nearly coincide — and, more to the point, node stays *above* weight's constant $M^\ast$ at every point on the path. The large ($\sim\!5\times$) reference under-count appears at sites where $\beta$ falls from $1$ toward $0$ (as in the Spine notebook); the code prints both so you can see which regime a site is in.

In [ ]:
c = target_a/(1-target_a)
# reconstruct dbar over the tracked trajectory
traj = [(s, (1-b)*r + b*H) for s,b in beta_traj]
Mn_traj = [c*(T*db+1) for _,db in traj]
Mn_ref  = c*(T*dbar_tr+1)
avg_traj = np.mean(Mn_traj)
print("M_n(t) along the trajectory (probes/step):", [round(x) for x in Mn_traj])
print(f"budget priced at reference beta   : {Mn_ref:.0f} probes/step")
print(f"budget integrated over beta(t)    : {avg_traj:.0f} probes/step (avg)")
print(f"-> reference pricing under-counts the early cost by {avg_traj/Mn_ref:.2f}x (large only if beta drops)")
# and the weight comparison
Mw = c*(Pw+1)
print(f"weight would need M* = {Mw:.0f} probes/step (constant)")
print(f"-> node/weight budget ratio along the path: {avg_traj/Mw:.2f}x  ({'node cheaper' if avg_traj<Mw else 'node MORE expensive at this site'})")

## What Tier 3 establishes — honestly

**The node-ZO advantage does not transfer for free to a per-token backbone.** The effective dimension there is $D_\text{node}=T\bar d$, so:

- At the standard LoRA init ($B=0$, $\beta=1$) $\bar d=m$ and $D_\text{node}=T m$ — **no advantage**, and with token multiplicity it can exceed $P$.
- Through training $\beta$ stays high in this site, so node ZO remains at/above $P$ — it is **not cheaper** here.
- The advantage reappears only when $T\bar d < P$, i.e. small $\beta$ *and/or* large rank (the crossover plot). The MNIST advantage (Tiers 1–2) held because those sites were effectively single-token ($T=1$), where $\bar d\le m<P$ always.

**Consequences for the program (state these in the paper's caveats).**
1. Node ZO is a property of *low-multiplicity, low-$\beta$* sites — not a universal win. Choosing *which* layer to adapt is now a first-class decision.
2. The budget must be integrated over $\beta(t)$; reference-$\beta$ pricing under-counts the early cost.
3. A promising direction is to reduce multiplicity (pool tokens for the probe, or adapt a low-token/global site), or to force $\beta$ down (e.g. non-zero $B$ init), so that $T\bar d<P$ by construction.

**Falsifiers.** If node $D_\text{eff}$ does not track $T\bar d$ (check the finite-$\mu$ calibration), or if the headroom/base asserts trip, the run is not trustworthy. If a site is found where trained $\beta$ is small and $T\bar d<P$, node ZO should — and must — beat weight ZO there; that is the constructive test for Tier 4/5.